In [ ]:
import pandas as pd

expr_path = "/home/exx/projects/HEST/scripts/dataset_creator/label_matrix_top100_variable_genes.csv"

# Load with cell_id as index
expr = pd.read_csv(expr_path)
expr = expr.set_index("cell_id")

gene_cols = expr.columns  # the 100 genes

# Per-cell QC metrics
cell_total = expr[gene_cols].sum(axis=1)
cell_nz = (expr[gene_cols] > 0).sum(axis=1)

# Heuristic thresholds – adjust after inspecting histograms
min_total = 20
min_genes = 3

low_total = cell_total < min_total
low_genes = cell_nz < min_genes
low_quality_mask = low_total | low_genes

print("Total cells:", len(expr))
print("Low-quality cells:", low_quality_mask.sum())

low_quality_cells = expr.index[low_quality_mask]
good_cells = expr.index[~low_quality_mask]
expr_qc = expr.loc[good_cells].copy()
import numpy as np
genes = expr_qc.columns  # after QC
lib_size = expr_qc[genes].sum(axis=1)
lib_size_safe = lib_size.replace(0, np.nan)
scale_factor = 1e4  # counts per 10k
expr_norm = expr_qc[genes].div(lib_size_safe, axis=0) * scale_factor
expr_norm = expr_norm.fillna(0.0)
expr_log = np.log1p(expr_norm)
gene_mean = expr_log.mean(axis=0)
gene_std = expr_log.std(axis=0, ddof=0)
gene_std_safe = gene_std.replace(0, 1.0)

expr_z = (expr_log - gene_mean) / gene_std

In [ ]:
expr_z.to_csv("expr_z.csv")

In [ ]:
import numpy as np

genes = expr_qc.columns  # after QC
lib_size = expr_qc[genes].sum(axis=1)
lib_size_safe = lib_size.replace(0, np.nan)
scale_factor = 1e4  # counts per 10k
expr_norm = expr_qc[genes].div(lib_size_safe, axis=0) * scale_factor
expr_norm = expr_norm.fillna(0.0)
expr_log = np.log1p(expr_norm)
gene_mean = expr_log.mean(axis=0)
gene_std = expr_log.std(axis=0, ddof=0)

# Protect against std = 0
gene_std_safe = gene_std.replace(0, 1.0)

expr_z = (expr_log - gene_mean) / gene_std

In [ ]:
expr_log = np.log1p(expr_norm)
gene_mean = expr_log.mean(axis=0)
gene_std = expr_log.std(axis=0, ddof=0)

# Protect against std = 0
gene_std_safe = gene_std.replace(0, 1.0)

expr_z = (expr_log - gene_mean) / gene_std

In [ ]:
def inverse_transform(pred_z: np.ndarray, gene_mean: pd.Series, gene_std: pd.Series):
    pred_log = pred_z * gene_std.values + gene_mean.values
    pred_norm = np.expm1(pred_log)
    return pred_norm  # still normalized counts per 10k

In [ ]:
norm = inverse_transform(expr_z, gene_mean, gene_std)
norm

In [ ]:
# Convert normalized counts back to raw counts using the original library sizes
expr_counts = (
    pd.DataFrame(norm, index=expr_qc.index, columns=genes)
    .mul(lib_size_safe, axis=0)
    / scale_factor
).fillna(0.0)
expr_counts


In [ ]:
import pandas as pd

path = "/data/datasets/hest/expression_long/NCBI884_cell_gene_counts_long.parquet"
df_long = pd.read_parquet(path)
df_long = df_long[df_long["cell_id"].notna()]
df_long = df_long[df_long["cell_id"] != "UNASSIGNED"]
print(df_long.head(5))

              cell_uid slide_id     cell_id feature_name  count
83  NCBI884_aaaaljek-1  NCBI884  aaaaljek-1      SPARCL1      1
86  NCBI884_aaaaljge-1  NCBI884  aaaaljge-1         MEG3      2
91  NCBI884_aaaaljge-1  NCBI884  aaaaljge-1        HMOX1      1
93  NCBI884_aaaaljch-1  NCBI884  aaaaljch-1         FCN3      1
94  NCBI884_aaaaljge-1  NCBI884  aaaaljge-1       COL3A1      1


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# Slides available in this dataset
SLIDES = [
    "NCBI884", "NCBI883", "NCBI882", "NCBI881", "NCBI880",
    "NCBI879", "NCBI876", "NCBI875", "NCBI873", "NCBI870",
    "NCBI867", "NCBI866", "NCBI865", "NCBI864", "NCBI861",
    "NCBI860", "NCBI859", "NCBI858", "NCBI857", "NCBI856",
]

# Define slide-level splits
TEST_SLIDES = set()
VAL_SLIDES = {"NCBI856"}
TRAIN_SLIDES = set(SLIDES) - TEST_SLIDES - VAL_SLIDES

print("Train slides:", sorted(TRAIN_SLIDES))
print("Val slides:", sorted(VAL_SLIDES))
print("Test slides:", sorted(TEST_SLIDES))

# 1) Build mapping (cell_id, slide_id) from segmentation parquet files
rows = []
for slide_id in SLIDES:
    seg_path = Path(f"/data/datasets/hest/xenium_seg/{slide_id}_xenium_nucleus_seg.parquet")
    if not seg_path.is_file():
        raise FileNotFoundError(f"Segmentation file not found: {seg_path}")
    seg_df = pd.read_parquet(seg_path)

    tmp = pd.DataFrame({"cell_id": seg_df.index.astype(str)})
    tmp["slide_id"] = slide_id
    tmp["cell_uid"] = tmp["slide_id"].astype(str) + "_" + tmp["cell_id"].astype(str)
    rows.append(tmp)

cell_to_slide = pd.concat(rows, axis=0, ignore_index=True)
print("Total rows in segmentation mapping:", len(cell_to_slide))

# 2) Load expression counts (top 100 HVGs) and attach slide_id/cell_uid
expr_path = Path("/home/exx/Desktop/projects/MAD/DINO/dinov2/label_matrix_top100_variable_genes.csv")
expr = pd.read_csv(expr_path)
expr["cell_id"] = expr["cell_id"].astype(str)
print("Expression matrix shape (raw):", expr.shape)

# Merge keeps only cells with known slide_id.
# If a cell_id exists in multiple slides, this will expand into multiple rows (as intended).
expr = expr.merge(cell_to_slide, on="cell_id", how="inner")
print("Expression matrix after attaching slide info:", expr.shape)

# Use slide-aware id as the index for downstream training
expr = expr.set_index("cell_uid")
expr.index.name = "cell_uid"

# 3) Assign split per slide
def _assign_split(slide_id: str) -> str:
    if slide_id in TEST_SLIDES:
        return "test"
    if slide_id in VAL_SLIDES:
        return "val"
    return "train"

expr["split"] = expr["slide_id"].map(_assign_split)

# 4) QC and library-size normalization
gene_cols = [c for c in expr.columns if c not in ["cell_id", "slide_id", "split"]]

cell_total = expr[gene_cols].sum(axis=1)
cell_nz = (expr[gene_cols] > 0).sum(axis=1)

min_total = 20
min_genes = 3
low_quality_mask = (cell_total < min_total) | (cell_nz < min_genes)

print("Total rows before QC:", len(expr))
print("Low-quality rows removed:", int(low_quality_mask.sum()))

expr_qc = expr.loc[~low_quality_mask].copy()
print("Rows after QC:", len(expr_qc))

# Library-size normalize to counts per 10k
genes = gene_cols
lib_size = expr_qc[genes].sum(axis=1)
lib_size_safe = lib_size.replace(0, np.nan)
scale_factor = 1e4
expr_norm = expr_qc[genes].div(lib_size_safe, axis=0) * scale_factor
expr_norm = expr_norm.fillna(0.0)

# 5) Log1p transform and compute z-score stats using TRAIN slides only
expr_log = np.log1p(expr_norm)
train_mask = expr_qc["split"] == "train"
expr_log_train = expr_log.loc[train_mask]

print("Rows used to compute z-score stats (train only):", expr_log_train.shape[0])

gene_mean = expr_log_train.mean(axis=0)
gene_std = expr_log_train.std(axis=0, ddof=0)
gene_std_safe = gene_std.replace(0, 1.0)

expr_z = (expr_log - gene_mean) / gene_std_safe

# 6) Combine metadata + z-scored expression
out = expr_qc[["cell_id", "slide_id", "split"]].join(expr_z)
print("Final expr_z_all shape:", out.shape)
print("Index name:", out.index.name)

out_path_parquet = Path("/home/exx/Desktop/projects/MAD/DINO/dinov2/dinov2/finetune/expr_z_all.parquet")
out.to_parquet(out_path_parquet)
print("Wrote:", out_path_parquet)

Train slides: ['NCBI857', 'NCBI858', 'NCBI859', 'NCBI860', 'NCBI861', 'NCBI864', 'NCBI865', 'NCBI866', 'NCBI867', 'NCBI870', 'NCBI873', 'NCBI875', 'NCBI876', 'NCBI879', 'NCBI880', 'NCBI881', 'NCBI882', 'NCBI883', 'NCBI884']
Val slides: ['NCBI856']
Test slides: []
Total rows in segmentation mapping: 1044961
Expression matrix shape (raw): (46276, 101)
Expression matrix after attaching slide info: (728677, 103)
Total rows before QC: 728677
Low-quality rows removed: 2168
Rows after QC: 726509
Rows used to compute z-score stats (train only): 680376
Final expr_z_all shape: (726509, 103)
Index name: cell_uid
Wrote: /home/exx/Desktop/projects/MAD/DINO/dinov2/dinov2/finetune/expr_z_all.parquet


In [ ]:
# NCBI864
seg_path = Path(f"/data/datasets/hest/transcripts/NCBI783_transcripts.parquet")
seg_df = pd.read_parquet(seg_path)
seg_df

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance,geometry,he_x,he_y
0,281474976710656,b'aagigajm-1',0,b'RUNX1',15.260350,486.198669,8.657274,37.950714,b'A1',0.000000,b'\x01\x01\x00\x00\x00\xc3H\x84\x81\xb7\x0f\xd...,18494.867280,4866.680232
1,281474976710657,b'aaglehma-1',1,b'RUNX1',15.751765,566.889465,9.142598,19.262161,b'A1',0.000000,"b'\x01\x01\x00\x00\x00\xfb""\xe3\x13$\xc6\xd1@f...",18200.563714,4868.590682
2,281474976710658,b'aahpibdb-1',0,b'TCIM',18.547781,813.857788,13.415476,40.000000,b'A1',2.795314,b'\x01\x01\x00\x00\x00xP\xa6\xfb\xbd\xe4\xd0@\...,17298.968484,4879.829254
3,281474976710659,b'aagnjbkb-1',0,b'LUM',17.891180,425.405334,8.532911,40.000000,b'A1',1.741180,b'\x01\x01\x00\x00\x00\x9cc\xf0\x974G\xd2@\xdf...,18716.821774,4876.253289
4,281474976710660,b'aahfmlaj-1',0,b'LUM',20.031204,299.748718,9.812729,40.000000,b'A1',5.207764,b'\x01\x01\x00\x00\x00\xbe\xeec\x07\xbe\xb9\xd...,19174.969201,4883.839602
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15643049,281779919692274,b'odikggam-1',1,b'APOBEC3B',5439.949219,4692.814453,65.748428,31.568277,b'F9',0.000000,b'\x01\x01\x00\x00\x00\xdd\x9c\xb3`*\x92\xa8@]...,3145.082769,24676.026767
15643050,281779919692275,b'oegnmipl-1',1,b'NARS',5439.662109,4980.534668,67.963600,13.145605,b'F9',0.000000,b'\x01\x01\x00\x00\x00\xc9{\x0c\xa07X\xa0@`\x9...,2092.108643,24673.697131
15643051,281779919692277,b'oehacfli-1',1,b'TUBB2B',5439.668457,4991.947266,63.720261,16.255787,b'F9',0.000000,b'\x01\x01\x00\x00\x00\xe3\xebK\xb9\xd6\t\xa0@...,2052.919382,24675.284470
15643052,281779919692285,b'oehphnle-1',0,b'LUM',5439.736328,5037.002930,64.713005,15.570970,b'F9',0.769979,b'\x01\x01\x00\x00\x00 Z\xfe \xe4\x83\x9d@)\x9...,1888.972782,24675.576658


In [3]:
expr_path = Path("/home/exx/Desktop/projects/MAD/DINO/dinov2/label_matrix_top100_variable_genes.csv")
expr = pd.read_csv(expr_path)
expr

,cell_id,LYZ,VIM,HLA-DRA,SFTPC,FN1,EPAS1,CCN2,LUM,PTGDS,...,CD14,MUC5B,ITGA3,WFDC2,GZMA,IRF1,RTKN2,UBE2J1,WWTR1,CD34
0,aaaaaaab-1,5,21,9,3,11,25,1,2,7,...,1,0,2,0,0,1,1,1,1,2
1,aaaaaaac-1,45,43,29,4,34,1,0,0,0,...,1,0,0,0,0,0,1,5,1,1
2,aaaaaaad-1,70,70,63,0,49,0,0,0,0,...,7,0,1,0,0,0,0,2,0,0
3,aaaaaaae-1,69,66,54,2,42,2,0,0,0,...,3,0,0,0,0,2,0,0,0,0
4,aaaaaaaf-1,2,3,2,5,1,1,0,0,1,...,1,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46271,aaaalemb-1,2,14,7,1,4,0,1,5,9,...,0,0,0,0,0,0,0,0,0,0
46272,aaaalemc-1,6,17,7,0,1,2,1,9,10,...,0,0,2,0,1,0,5,1,1,0
46273,aaaalemd-1,0,27,3,0,1,14,2,15,19,...,0,0,2,0,0,0,0,0,7,1
46274,aaaaleme-1,2,10,10,0,1,9,0,2,2,...,0,0,3,0,0,0,4,1,8,2


In [7]:
import h5py
import pandas as pd
from pathlib import Path

# Slide-aware labels from step (2); index should be cell_uid like "NCBI856_<cell_id>"
labels_path = Path("/home/exx/Desktop/projects/MAD/DINO/dinov2/dinov2/finetune/expr_z_all.parquet")

# Slide-aware combined morphology H5 built with:
# combine_h5_shards.py --prefix-from stem
morph_h5_path = Path("/data/datasets/MAD/HEST_h5/morphology/ALL_NCBI_morphology_slideaware.h5")

def _decode(x):
    return x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else str(x)

with h5py.File(morph_h5_path, "r") as h5f:
    filenames = [_decode(fn) for fn in h5f["filenames"][:]]

# H5 stems should be like "NCBI856_aaaaaaab-1"
valid_ids = {name.split(".", 1)[0] for name in filenames}
print("Total H5 entries:", len(filenames))
print("Unique ids in H5 (stems):", len(valid_ids))

labels_full = pd.read_parquet(labels_path)
print("Labels full shape:", labels_full.shape)
print("Labels index name:", labels_full.index.name)

labels_compact = labels_full.loc[labels_full.index.intersection(valid_ids)].copy()
print("Labels compact shape:", labels_compact.shape)

if "split" in labels_compact.columns:
    print("Splits in compact labels:")
    print(labels_compact["split"].value_counts(dropna=False))

out_path = labels_path.with_name("expr_z_hest_slideaware.parquet")
labels_compact.to_parquet(out_path)
print("Wrote:", out_path)

Total H5 entries: 1044961
Unique ids in H5 (stems): 1044961
Labels full shape: (726509, 103)
Labels index name: cell_uid
Labels compact shape: (726509, 103)
Splits in compact labels:
split
train    680376
val       46133
Name: count, dtype: int64
Wrote: /home/exx/Desktop/projects/MAD/DINO/dinov2/dinov2/finetune/expr_z_hest_slideaware.parquet


In [ ]:
# Build slide-aware compact labels matched to a slide-aware combined H5
# Expected:
# - labels_path (expr_z_all.parquet) indexed by cell_uid like "NCBI856_<cell_id>"
# - morph_h5_path filenames like "NCBI856_<cell_id>.png"

import h5py
import pandas as pd
from pathlib import Path

labels_path = Path("/home/exx/Desktop/projects/MAD/DINO/dinov2/dinov2/finetune/expr_z_all.parquet")
# Point this to your slide-aware combined morphology H5 (created with combine_h5_shards.py --prefix-from stem)
morph_h5_path = Path("/data/datasets/MAD/HEST_h5/morphology/ALL_NCBI_morphology_slideaware.h5")


def _decode(x):
    return x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else str(x)


with h5py.File(morph_h5_path, "r") as h5f:
    filenames = [_decode(fn) for fn in h5f["filenames"][:]]

valid_ids = {name.split(".")[0] for name in filenames}  # stems like NCBI856_aaaaaaab-1
print("Total H5 entries:", len(filenames))
print("Unique ids in H5 (stems):", len(valid_ids))

labels_full = pd.read_parquet(labels_path)
print("Labels full shape:", labels_full.shape)
print("Labels index name:", labels_full.index.name)

labels_compact = labels_full.loc[labels_full.index.intersection(valid_ids)].copy()
print("Labels compact shape:", labels_compact.shape)
if "split" in labels_compact.columns:
    print("Splits in compact labels:")
    print(labels_compact["split"].value_counts(dropna=False))

compact_path = labels_path.with_name("expr_z_hest_slideaware.parquet")
labels_compact.to_parquet(compact_path)
print("Wrote compact labels to:", compact_path)


In [ ]:
import h5py
import pandas as pd
from pathlib import Path
labels_path = Path("/home/exx/Desktop/projects/MAD/DINO/dinov2/dinov2/finetune/expr_z_all.parquet")
labels_full = pd.read_parquet(labels_path)
print("Labels full shape:", labels_full.shape)
print("Labels index name:", labels_full.index.name)

Labels full shape: (726509, 103)
Labels index name: cell_uid


In [2]:
labels_full

,cell_id,slide_id,split,LYZ,VIM,HLA-DRA,SFTPC,FN1,EPAS1,CCN2,...,CD14,MUC5B,ITGA3,WFDC2,GZMA,IRF1,RTKN2,UBE2J1,WWTR1,CD34
cell_uid,,,,,,,,,,,,,,,,,,,,,
NCBI884_aaaaaaab-1,aaaaaaab-1,NCBI884,train,0.729793,0.481025,0.276488,0.928093,0.766810,1.020250,-0.083771,...,1.301833,-0.216573,1.174590,-0.492509,-0.421435,0.875292,1.469659,0.605065,0.63791,1.501415
NCBI883_aaaaaaab-1,aaaaaaab-1,NCBI883,train,0.729793,0.481025,0.276488,0.928093,0.766810,1.020250,-0.083771,...,1.301833,-0.216573,1.174590,-0.492509,-0.421435,0.875292,1.469659,0.605065,0.63791,1.501415
NCBI882_aaaaaaab-1,aaaaaaab-1,NCBI882,train,0.729793,0.481025,0.276488,0.928093,0.766810,1.020250,-0.083771,...,1.301833,-0.216573,1.174590,-0.492509,-0.421435,0.875292,1.469659,0.605065,0.63791,1.501415
NCBI881_aaaaaaab-1,aaaaaaab-1,NCBI881,train,0.729793,0.481025,0.276488,0.928093,0.766810,1.020250,-0.083771,...,1.301833,-0.216573,1.174590,-0.492509,-0.421435,0.875292,1.469659,0.605065,0.63791,1.501415
NCBI880_aaaaaaab-1,aaaaaaab-1,NCBI880,train,0.729793,0.481025,0.276488,0.928093,0.766810,1.020250,-0.083771,...,1.301833,-0.216573,1.174590,-0.492509,-0.421435,0.875292,1.469659,0.605065,0.63791,1.501415
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NCBI866_aaaalemf-1,aaaalemf-1,NCBI866,train,0.165843,0.227736,-0.054953,1.132585,-0.197448,0.790455,-1.622851,...,-0.610907,-0.216573,1.196833,-0.492509,2.712251,1.200867,-0.533112,0.627277,-1.01917,-0.631824
NCBI865_aaaalemf-1,aaaalemf-1,NCBI865,train,0.165843,0.227736,-0.054953,1.132585,-0.197448,0.790455,-1.622851,...,-0.610907,-0.216573,1.196833,-0.492509,2.712251,1.200867,-0.533112,0.627277,-1.01917,-0.631824
NCBI864_aaaalemf-1,aaaalemf-1,NCBI864,train,0.165843,0.227736,-0.054953,1.132585,-0.197448,0.790455,-1.622851,...,-0.610907,-0.216573,1.196833,-0.492509,2.712251,1.200867,-0.533112,0.627277,-1.01917,-0.631824


In [5]:
import h5py
import pandas as pd
from pathlib import Path

# Paths
labels_path = Path("/home/exx/Desktop/projects/MAD/DINO/dinov2/dinov2/finetune/expr_z_all.parquet")
morph_h5_path = Path("/data/datasets/MAD/HEST_h5/morphology/ALL_NCBI_morphology.h5")

# 1) Collect all cell_ids that actually have images in the combined morph H5
def _decode(x):
    return x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else str(x)

with h5py.File(morph_h5_path, "r") as h5f:
    filenames = [_decode(fn) for fn in h5f["filenames"][:]]

cell_ids_from_h5 = [name.split(".")[0] for name in filenames]
valid_ids = set(cell_ids_from_h5)
print("Total H5 entries:", len(filenames))
print("Unique cell_ids in H5:", len(valid_ids))

# 2) Load full labels (with slide_id + split) and filter to those ids
labels_full = pd.read_parquet(labels_path)
print("Labels full shape:", labels_full.shape)

labels_compact = labels_full.loc[labels_full.index.intersection(valid_ids)].copy()
# Drop duplicate index entries if any
labels_compact = labels_compact[~labels_compact.index.duplicated(keep="first")]
print("Labels compact shape:", labels_compact.shape)
print("Splits in compact labels:", labels_compact["split"].value_counts())

# 3) Save compact labels for training
compact_path = labels_path.with_name("expr_z_hest.parquet")
labels_compact.to_parquet(compact_path)
print("Wrote compact labels to:", compact_path)

Total H5 entries: 1044961
Unique cell_ids in H5: 140997
Labels full shape: (12200961, 102)
Labels compact shape: (46133, 102)
Splits in compact labels: split
train    46133
Name: count, dtype: int64
Wrote compact labels to: /home/exx/Desktop/projects/MAD/DINO/dinov2/dinov2/finetune/expr_z_hest.parquet
